# DE2 — Final Project Notebook
> **Track D : Aviation / OpenSky Network**  
> Author : Badr TAJINI — Data Engineering II (Data-Intensive Workloads) — ESIEE 2025-2026

---

## Pipeline Overview

| Stage | Description | SLO |
|-------|-------------|-----|
| 1. Bronze | Land raw OpenSky CSV immutably | — |
| 2. Silver | Clean, type-cast, deduplicate, schema contracts | — |
| 3. Gold | Analytics tables, partitioned Parquet | bronze→gold ≤ 10 min |
| 4. Streaming | Windowed flight aggregations via Structured Streaming | trigger ≤ 30 s |
| 5. Text | Inverted index on callsign/origin, TF-IDF features | query ≤ 2 s |
| 6. Iterative | KMeans clustering sweep on flight features | silhouette ≥ 0.25 |
| 7. LLM Prep | Curated text corpus with quality filters | ≥ 80% pass rate |
| 8. Evidence | Plans, metrics CSV, Spark UI screenshots | — |

**Dataset:** OpenSky Network ADS-B data — `flights_data4.csv` (or equivalent bulk export)  
**Iterative workload choice:** Clustering (KMeans + BisectingKMeans sweep)

---
## 0. Setup — Config, Spark, Helpers

In [17]:
import os

os.environ["PYSPARK_PYTHON"] = "/home/maxence/miniconda3/envs/de1-env/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/home/maxence/miniconda3/envs/de1-env/bin/python"

In [18]:
import yaml, pathlib, datetime, time, json, os, hashlib
import pandas as pd
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

# ── Load config ──────────────────────────────────────────────────────────────
with open("de2_project_config.yml") as fh:
    CFG = yaml.safe_load(fh)

# ── Spark session ─────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("de2-project-aviation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")        # low for laptop
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")       # AQE on
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark UI     :", spark.sparkContext.uiWebUrl)
print()
print(json.dumps(CFG, indent=2, default=str))

Spark version: 4.0.1
Spark UI     : http://max-HP-Laptop-14s-dq2xxx.lan:4040

{
  "track": "aviation_opensky",
  "paths": {
    "raw_csv_glob": "data/project/raw/*.csv",
    "bronze": "outputs/project/bronze",
    "silver": "outputs/project/silver",
    "gold": "outputs/project/gold",
    "streaming_landing": "data/project/landing/",
    "text_corpus": "data/project/raw/*.csv",
    "proof": "proof/",
    "llm_ready": "outputs/project/llm_ready"
  },
  "layout": {
    "partition_by": [
      "icao24",
      "callsign"
    ],
    "compaction_target_mb": 128
  },
  "streaming": {
    "watermark": "30 minutes",
    "window_duration": "10 minutes",
    "trigger_interval": "30 seconds",
    "timeout_seconds": 90
  },
  "text": {
    "min_token_length": 2,
    "query_terms": [
      "flight",
      "airport",
      "delay",
      "arrival",
      "departure"
    ],
    "stopwords": [
      "the",
      "a",
      "an",
      "is",
      "are",
      "was",
      "were",
      "in",
      "on"

In [3]:
# ── Helper — metrics log ──────────────────────────────────────────────────────
METRICS_FILE = "project_metrics_log.csv"
METRICS_COLS = ["run_id", "stage", "task", "metric_name", "metric_value", "notes", "timestamp"]
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

def log_metric(stage, task, metric_name, metric_value, notes=""):
    row = {
        "run_id": RUN_ID,
        "stage": stage,
        "task": task,
        "metric_name": metric_name,
        "metric_value": metric_value,
        "notes": notes,
        "timestamp": datetime.datetime.now().isoformat()
    }
    if pathlib.Path(METRICS_FILE).exists():
        df = pd.read_csv(METRICS_FILE)
    else:
        df = pd.DataFrame(columns=METRICS_COLS)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(METRICS_FILE, index=False)
    print(f"  [METRIC] {stage}/{task}: {metric_name} = {metric_value}  ({notes})")

def save_plan(df, path):
    """Save EXPLAIN FORMATTED to a text file in proof/."""
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as fh:
        fh.write(df._jdf.queryExecution().toString())
    print(f"  [PLAN] Saved → {path}")

def parquet_size_mb(path):
    """Compute total size of a Parquet folder in MB."""
    total = 0
    for p in pathlib.Path(path).rglob("*.parquet"):
        total += p.stat().st_size
    return round(total / 1_048_576, 2)

def csv_size_mb(glob_path):
    import glob as g
    total = sum(pathlib.Path(f).stat().st_size for f in g.glob(glob_path))
    return round(total / 1_048_576, 2)

# Create output dirs
for d in ["outputs/project/bronze", "outputs/project/silver", "outputs/project/gold",
          "outputs/project/streaming", "outputs/project/text", "outputs/project/llm_ready",
          "data/project/raw", "data/project/landing", "proof"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Helpers loaded. Run ID:", RUN_ID)

Helpers loaded. Run ID: 20260524_192827


---
## 0b. Data Acquisition

**Option A (preferred):** Download a real OpenSky bulk CSV from https://zenodo.org/records/7923702  
Place the file(s) in `data/project/raw/`.

**Option B (used here):** Generate a realistic synthetic dataset (≥ 10 M rows when looped) that mirrors the exact OpenSky schema. Useful for local runs when Zenodo is unavailable.

In [4]:
import glob as g

existing_raw = g.glob(CFG["paths"]["raw_csv_glob"])
print(f"Existing raw files: {len(existing_raw)}")

if not existing_raw:
    print("No raw data found — generating synthetic OpenSky dataset...")
    import random, math
    import numpy as np

    np.random.seed(42)
    random.seed(42)

    # OpenSky schema: icao24, callsign, estdepartureairport, estarrivalairport,
    # firstseen, lastseen, day, estdepartureairporthorizdistance,
    # estdepartureairportvertdistance, estarrivalairporthorizdistance,
    # estarrivalairportvertdistance, departureairportcandidatescount,
    # arrivalairportcandidatescount

    airports = ["LFPG", "EGLL", "LEMD", "LIRF", "EDDF", "EHAM", "LEBL",
                "EDDM", "EGKK", "LFPO", "LOWW", "LSZH", "LPPT", "EKCH",
                "EFHK", "ENGM", "ESSA", "EPWA", "LKPR", "LHBP",
                "KJFK", "KLAX", "KORD", "KATL", "KDFW", "KDEN",
                "VHHH", "RJTT", "WSSS", "YSSY", "OMDB", "OTHH"]

    manufacturers = ["Boeing", "Airbus", "Bombardier", "Embraer"]
    models_map = {
        "Boeing":    ["737", "747", "777", "787"],
        "Airbus":    ["A320", "A321", "A330", "A350", "A380"],
        "Bombardier":["CRJ200", "CRJ900", "Q400"],
        "Embraer":   ["E175", "E190", "E195"]
    }
    airlines = ["AFR", "BAW", "DLH", "EZY", "RYR", "UAE", "QTR",
                "AAL", "UAL", "DAL", "SWA", "ACA", "SIA", "QFA"]

    BASE_TIME = int(datetime.datetime(2023, 1, 1).timestamp())
    N_ROWS_PER_FILE = 1_000_000  # 1M rows per file → generate multiple files for ≥10M
    N_FILES = 1  # increase to 10+ for full 10M requirement; 1 suffices for local dev

    for file_idx in range(N_FILES):
        rows = []
        for _ in range(N_ROWS_PER_FILE):
            mfr = random.choice(manufacturers)
            model = random.choice(models_map[mfr])
            airline = random.choice(airlines)
            dep = random.choice(airports)
            arr = random.choice([a for a in airports if a != dep])
            day_offset = random.randint(0, 364)
            firstseen = BASE_TIME + day_offset * 86400 + random.randint(0, 72000)
            flight_dur = random.randint(1800, 54000)  # 30 min – 15 h
            lastseen = firstseen + flight_dur
            day_ts = BASE_TIME + day_offset * 86400
            icao24 = ''.join(random.choices('0123456789abcdef', k=6))
            callsign = f"{airline}{random.randint(100, 9999)}"
            dep_horiz = random.randint(0, 50000)
            dep_vert  = random.randint(0, 10000)
            arr_horiz = random.randint(0, 50000)
            arr_vert  = random.randint(0, 10000)
            dep_cand  = random.randint(1, 5)
            arr_cand  = random.randint(1, 5)

            rows.append((
                icao24, callsign, dep, arr,
                firstseen, lastseen, day_ts,
                dep_horiz, dep_vert, arr_horiz, arr_vert,
                dep_cand, arr_cand
            ))

        cols = ["icao24","callsign","estdepartureairport","estarrivalairport",
                "firstseen","lastseen","day",
                "estdepartureairporthorizdistance","estdepartureairportvertdistance",
                "estarrivalairporthorizdistance","estarrivalairportvertdistance",
                "departureairportcandidatescount","arrivalairportcandidatescount"]

        df_gen = pd.DataFrame(rows, columns=cols)
        out_path = f"data/project/raw/opensky_2023_{file_idx:02d}.csv"
        df_gen.to_csv(out_path, index=False)
        print(f"  Generated {len(df_gen):,} rows → {out_path}")
        del df_gen, rows

    print("Synthetic data generation done.")
else:
    print(f"Using existing raw files: {existing_raw[:5]}")

Existing raw files: 1
Using existing raw files: ['data/project/raw/opensky_2023_00.csv']


---
## 1. Bronze — Landing Raw Data Immutably

Raw CSVs land in `bronze/` as-is (append-only). No transformations.  
Schema: all strings at bronze layer — no coercions.

In [5]:
t0 = time.time()
raw_glob = CFG["paths"]["raw_csv_glob"]
bronze   = CFG["paths"]["bronze"]
proof    = CFG["paths"]["proof"]

# ── Read raw CSVs (all strings — bronze is immutable)
BRONZE_SCHEMA = T.StructType([
    T.StructField("icao24",                              T.StringType(), True),
    T.StructField("callsign",                            T.StringType(), True),
    T.StructField("estdepartureairport",                 T.StringType(), True),
    T.StructField("estarrivalairport",                   T.StringType(), True),
    T.StructField("firstseen",                           T.StringType(), True),
    T.StructField("lastseen",                            T.StringType(), True),
    T.StructField("day",                                 T.StringType(), True),
    T.StructField("estdepartureairporthorizdistance",     T.StringType(), True),
    T.StructField("estdepartureairportvertdistance",      T.StringType(), True),
    T.StructField("estarrivalairporthorizdistance",       T.StringType(), True),
    T.StructField("estarrivalairportvertdistance",        T.StringType(), True),
    T.StructField("departureairportcandidatescount",      T.StringType(), True),
    T.StructField("arrivalairportcandidatescount",        T.StringType(), True),
])

df_bronze = (
    spark.read
    .schema(BRONZE_SCHEMA)
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .csv(raw_glob)
)

n_bronze = df_bronze.count()
df_bronze.write.mode("overwrite").parquet(bronze)   # Parquet at bronze for footprint

duration_bronze = round(time.time() - t0, 2)
size_bronze_mb  = parquet_size_mb(bronze)
size_raw_mb     = csv_size_mb(raw_glob)

print(f"Bronze rows     : {n_bronze:,}")
print(f"Bronze Parquet  : {size_bronze_mb} MB")
print(f"Raw CSV         : {size_raw_mb} MB")
print(f"Ingestion time  : {duration_bronze} s")

log_metric("etl", "bronze", "row_count",       n_bronze,         "raw ingestion")
log_metric("etl", "bronze", "parquet_mb",       size_bronze_mb,   "bronze Parquet footprint")
log_metric("etl", "bronze", "csv_mb",           size_raw_mb,      "raw CSV baseline")
log_metric("etl", "bronze", "duration_sec",     duration_bronze,  "bronze write time")

df_bronze.printSchema()
df_bronze.show(5, truncate=False)

26/05/24 19:28:29 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/project/raw/*.csv.
java.io.FileNotFoundException: File data/project/raw/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resolve

Bronze rows     : 1,000,000
Bronze Parquet  : 44.72 MB
Raw CSV         : 79.39 MB
Ingestion time  : 6.37 s
  [METRIC] etl/bronze: row_count = 1000000  (raw ingestion)
  [METRIC] etl/bronze: parquet_mb = 44.72  (bronze Parquet footprint)
  [METRIC] etl/bronze: csv_mb = 79.39  (raw CSV baseline)
  [METRIC] etl/bronze: duration_sec = 6.37  (bronze write time)
root
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- estdepartureairport: string (nullable = true)
 |-- estarrivalairport: string (nullable = true)
 |-- firstseen: string (nullable = true)
 |-- lastseen: string (nullable = true)
 |-- day: string (nullable = true)
 |-- estdepartureairporthorizdistance: string (nullable = true)
 |-- estdepartureairportvertdistance: string (nullable = true)
 |-- estarrivalairporthorizdistance: string (nullable = true)
 |-- estarrivalairportvertdistance: string (nullable = true)
 |-- departureairportcandidatescount: string (nullable = true)
 |-- arrivalairportcandidates

---
## 2. Silver — Cleaning, Typing, Schema Contracts, Deduplication

Contracts enforced:
- `icao24` — NOT NULL, 6-char hex
- `firstseen`, `lastseen`, `day` — cast to LongType (UNIX epoch)
- `flight_duration_sec` — derived feature, must be > 0
- `callsign` — trimmed, upper-cased
- Deduplication on natural key `(icao24, firstseen)`

In [6]:
t0     = time.time()
silver = CFG["paths"]["silver"]

# ── Read bronze
df_b = spark.read.parquet(bronze)

# ── Type-cast & clean
df_silver = (
    df_b
    # Type casts
    .withColumn("firstseen",   F.col("firstseen").cast(T.LongType()))
    .withColumn("lastseen",    F.col("lastseen").cast(T.LongType()))
    .withColumn("day",         F.col("day").cast(T.LongType()))
    .withColumn("dep_horiz_m", F.col("estdepartureairporthorizdistance").cast(T.IntegerType()))
    .withColumn("dep_vert_m",  F.col("estdepartureairportvertdistance").cast(T.IntegerType()))
    .withColumn("arr_horiz_m", F.col("estarrivalairporthorizdistance").cast(T.IntegerType()))
    .withColumn("arr_vert_m",  F.col("estarrivalairportvertdistance").cast(T.IntegerType()))
    .withColumn("dep_cand",    F.col("departureairportcandidatescount").cast(T.IntegerType()))
    .withColumn("arr_cand",    F.col("arrivalairportcandidatescount").cast(T.IntegerType()))
    # Normalise strings
    .withColumn("icao24",   F.trim(F.lower(F.col("icao24"))))
    .withColumn("callsign", F.trim(F.upper(F.col("callsign"))))
    .withColumn("dep_airport", F.trim(F.upper(F.col("estdepartureairport"))))
    .withColumn("arr_airport", F.trim(F.upper(F.col("estarrivalairport"))))
    # Derived feature
    .withColumn("flight_duration_sec", F.col("lastseen") - F.col("firstseen"))
    .withColumn("event_ts",
        F.to_timestamp(F.col("firstseen").cast(T.LongType())))
    # ── Schema contracts
    .filter(F.col("icao24").isNotNull())
    .filter(F.length(F.col("icao24")) == 6)
    .filter(F.col("firstseen").isNotNull())
    .filter(F.col("lastseen").isNotNull())
    .filter(F.col("flight_duration_sec") > 0)
    .filter(F.col("dep_airport").isNotNull())
    .filter(F.col("arr_airport").isNotNull())
    .filter(F.col("dep_airport") != F.col("arr_airport"))
    # ── Deduplicate on natural key
    .dropDuplicates(["icao24", "firstseen"])
    # ── Select final silver columns
    .select(
        "icao24", "callsign", "dep_airport", "arr_airport",
        "firstseen", "lastseen", "day", "event_ts",
        "flight_duration_sec",
        "dep_horiz_m", "dep_vert_m", "arr_horiz_m", "arr_vert_m",
        "dep_cand", "arr_cand"
    )
)

# ── Save plan BEFORE write
save_plan(df_silver, f"{proof}/plan_silver.txt")

# ── Write silver
df_silver.write.mode("overwrite").parquet(silver)

n_silver       = df_silver.count()
size_silver_mb = parquet_size_mb(silver)
duration_silver = round(time.time() - t0, 2)

print(f"Silver rows      : {n_silver:,}")
print(f"Silver Parquet   : {size_silver_mb} MB")
print(f"Silver duration  : {duration_silver} s")
print(f"Rows dropped     : {n_bronze - n_silver:,}  ({100*(n_bronze-n_silver)/n_bronze:.1f}%)")

log_metric("etl", "silver", "row_count",     n_silver,       "after cleaning & dedup")
log_metric("etl", "silver", "parquet_mb",    size_silver_mb, "silver Parquet footprint")
log_metric("etl", "silver", "rows_dropped",  n_bronze - n_silver, "contract violations + dedup")
log_metric("etl", "silver", "duration_sec",  duration_silver)

df_silver.printSchema()
df_silver.show(5, truncate=False)

26/05/24 19:28:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  [PLAN] Saved → proof//plan_silver.txt


Silver rows      : 1,000,000
Silver Parquet   : 50.1 MB
Silver duration  : 9.23 s
Rows dropped     : 0  (0.0%)
  [METRIC] etl/silver: row_count = 1000000  (after cleaning & dedup)
  [METRIC] etl/silver: parquet_mb = 50.1  (silver Parquet footprint)
  [METRIC] etl/silver: rows_dropped = 0  (contract violations + dedup)
  [METRIC] etl/silver: duration_sec = 9.23  ()
root
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- dep_airport: string (nullable = true)
 |-- arr_airport: string (nullable = true)
 |-- firstseen: long (nullable = true)
 |-- lastseen: long (nullable = true)
 |-- day: long (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- flight_duration_sec: long (nullable = true)
 |-- dep_horiz_m: integer (nullable = true)
 |-- dep_vert_m: integer (nullable = true)
 |-- arr_horiz_m: integer (nullable = true)
 |-- arr_vert_m: integer (nullable = true)
 |-- dep_cand: integer (nullable = true)
 |-- arr_cand: integer (nullable = true)



+------+--------+-----------+-----------+----------+----------+----------+-------------------+-------------------+-----------+----------+-----------+----------+--------+--------+
|icao24|callsign|dep_airport|arr_airport|firstseen |lastseen  |day       |event_ts           |flight_duration_sec|dep_horiz_m|dep_vert_m|arr_horiz_m|arr_vert_m|dep_cand|arr_cand|
+------+--------+-----------+-----------+----------+----------+----------+-------------------+-------------------+-----------+----------+-----------+----------+--------+--------+
|000153|SWA8502 |WSSS       |OMDB       |1700496061|1700543961|1700434800|2023-11-20 17:01:01|47900              |9124       |4333      |13745      |4060      |1       |1       |
|000184|UAL4265 |LFPG       |OMDB       |1698820754|1698846701|1698793200|2023-11-01 07:39:14|25947              |33767      |1984      |4705       |5252      |4       |2       |
|0001f4|ACA8578 |KLAX       |EKCH       |1673979633|1674012741|1673910000|2023-01-17 19:20:33|33108      

---
## 3. Gold — Analytics Tables

Three analytics tables:
- **gold/routes** — aggregated route-level stats (partitioned by `dep_airport`)
- **gold/daily** — daily flight volume & avg duration (partitioned by `day_date`)
- **gold/aircraft** — per-aircraft career summary

Evidence: EXPLAIN plan saved, before/after Parquet footprint logged.

In [7]:
t0   = time.time()
gold = CFG["paths"]["gold"]

df_s = spark.read.parquet(silver)

# ── Gold table 1: route-level aggregations
gold_routes = (
    df_s
    .groupBy("dep_airport", "arr_airport")
    .agg(
        F.count("*").alias("n_flights"),
        F.avg("flight_duration_sec").alias("avg_duration_sec"),
        F.min("flight_duration_sec").alias("min_duration_sec"),
        F.max("flight_duration_sec").alias("max_duration_sec"),
        F.stddev("flight_duration_sec").alias("stddev_duration_sec"),
        F.countDistinct("icao24").alias("n_distinct_aircraft")
    )
    .withColumn("avg_duration_h", F.round(F.col("avg_duration_sec") / 3600, 2))
    .orderBy(F.desc("n_flights"))
)

save_plan(gold_routes, f"{proof}/plan_gold_routes.txt")
gold_routes.write.mode("overwrite").partitionBy("dep_airport").parquet(f"{gold}/routes")

# ── Gold table 2: daily flight volume
gold_daily = (
    df_s
    .withColumn("day_date", F.from_unixtime("day", "yyyy-MM-dd"))
    .groupBy("day_date")
    .agg(
        F.count("*").alias("n_flights"),
        F.avg("flight_duration_sec").alias("avg_duration_sec"),
        F.countDistinct("dep_airport").alias("n_dep_airports"),
        F.countDistinct("arr_airport").alias("n_arr_airports")
    )
    .orderBy("day_date")
)

save_plan(gold_daily, f"{proof}/plan_gold_daily.txt")
gold_daily.write.mode("overwrite").parquet(f"{gold}/daily")

# ── Gold table 3: per-aircraft career summary
gold_aircraft = (
    df_s
    .groupBy("icao24")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum("flight_duration_sec").alias("total_air_time_sec"),
        F.countDistinct("dep_airport").alias("n_airports_visited"),
        F.min("event_ts").alias("first_seen"),
        F.max("event_ts").alias("last_seen"),
        F.collect_set("callsign").alias("callsigns_used")
    )
    .withColumn("total_air_time_h", F.round(F.col("total_air_time_sec") / 3600, 2))
)

save_plan(gold_aircraft, f"{proof}/plan_gold_aircraft.txt")
gold_aircraft.write.mode("overwrite").parquet(f"{gold}/aircraft")

duration_gold = round(time.time() - t0, 2)
size_gold_mb  = parquet_size_mb(gold)
ratio_vs_csv  = round(size_gold_mb / size_raw_mb, 3) if size_raw_mb > 0 else None

print(f"Gold written in  : {duration_gold} s")
print(f"Gold total MB    : {size_gold_mb}")
print(f"Ratio vs raw CSV : {ratio_vs_csv}  (SLO ≤ 0.60)")
print(f"\nTop routes:")
gold_routes.show(10, truncate=False)

log_metric("etl", "gold", "duration_sec",    duration_gold)
log_metric("etl", "gold", "parquet_mb",       size_gold_mb)
log_metric("etl", "gold", "storage_ratio",    ratio_vs_csv,   f"SLO ≤ 0.60 → {'PASS' if ratio_vs_csv and ratio_vs_csv <= 0.60 else 'FAIL'}")

  [PLAN] Saved → proof//plan_gold_routes.txt


  [PLAN] Saved → proof//plan_gold_daily.txt


  [PLAN] Saved → proof//plan_gold_aircraft.txt


Gold written in  : 9.92 s
Gold total MB    : 35.62
Ratio vs raw CSV : 0.449  (SLO ≤ 0.60)

Top routes:


+-----------+-----------+---------+------------------+----------------+----------------+-------------------+-------------------+--------------+
|dep_airport|arr_airport|n_flights|avg_duration_sec  |min_duration_sec|max_duration_sec|stddev_duration_sec|n_distinct_aircraft|avg_duration_h|
+-----------+-----------+---------+------------------+----------------+----------------+-------------------+-------------------+--------------+
|KDEN       |EFHK       |1115     |27810.0           |1800            |53999           |15001.609328988843 |1115               |7.73          |
|KDFW       |LSZH       |1102     |27969.24500907441 |1807            |53921           |14902.158844933962 |1102               |7.77          |
|YSSY       |KDFW       |1096     |27543.49087591241 |1892            |53990           |14971.302635525166 |1096               |7.65          |
|EDDM       |KATL       |1092     |27950.25183150183 |1873            |53994           |15003.983169579382 |1092               |7.76    

---
## 4. Streaming Pipeline — Windowed Aggregation

**Design:** Simulates a live ADS-B feed by drip-feeding silver records as CSV files.  
**Watermark:** 30 min on `event_ts`  
**Window:** 10-min tumbling window on `dep_airport`  
**Sink:** Parquet in append mode  
**Evidence:** `query.lastProgress` → `proof/query_progress.json`

In [9]:
# ── Prepare landing zone — drip a small slice of silver as streaming source
landing = CFG["paths"].get("streaming_landing", "data/project/landing/")
pathlib.Path(landing).mkdir(parents=True, exist_ok=True)

STREAM_SCHEMA = T.StructType([
    T.StructField("icao24",              T.StringType(),  True),
    T.StructField("callsign",            T.StringType(),  True),
    T.StructField("dep_airport",         T.StringType(),  True),
    T.StructField("arr_airport",         T.StringType(),  True),
    T.StructField("firstseen",           T.LongType(),    True),
    T.StructField("lastseen",            T.LongType(),    True),
    T.StructField("day",                 T.LongType(),    True),
    T.StructField("event_ts",            T.TimestampType(),True),
    T.StructField("flight_duration_sec", T.LongType(),    True),
    T.StructField("dep_horiz_m",         T.IntegerType(), True),
    T.StructField("dep_vert_m",          T.IntegerType(), True),
    T.StructField("arr_horiz_m",         T.IntegerType(), True),
    T.StructField("arr_vert_m",          T.IntegerType(), True),
    T.StructField("dep_cand",            T.IntegerType(), True),
    T.StructField("arr_cand",            T.IntegerType(), True),
])

# Write 3 micro-batches into the landing zone (simulates arriving files)
df_feed = spark.read.parquet(silver).limit(30_000)
for i in range(3):
    batch = df_feed.limit(10_000).filter(F.col("firstseen") % 3 == i)
    batch_path = f"{landing}/batch_{i:03d}.csv"
    batch.toPandas().to_csv(batch_path, index=False)

print(f"Landing zone populated with {len(list(pathlib.Path(landing).glob('*.csv')))} files.")

Landing zone populated with 3 files.


In [10]:
# ── Structured Streaming pipeline
STREAM_OUT       = "outputs/project/streaming"
STREAM_CKPT      = "outputs/project/streaming_checkpoint"
pathlib.Path(STREAM_OUT).mkdir(parents=True, exist_ok=True)

df_stream = (
    spark.readStream
    .schema(STREAM_SCHEMA)
    .option("header", "true")
    .option("maxFilesPerTrigger", 1)
    .csv(landing)
)

# ── Windowed aggregation with watermark
windowed = (
    df_stream
    .withWatermark("event_ts", CFG["streaming"]["watermark"])
    .groupBy(
        F.window("event_ts", CFG["streaming"]["window_duration"]),
        "dep_airport"
    )
    .agg(
        F.count("*").alias("n_flights"),
        F.avg("flight_duration_sec").alias("avg_duration_sec"),
        F.approx_count_distinct("icao24").alias("n_aircraft")
    )
)

# ── Sink: append Parquet
query = (
    windowed.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", STREAM_OUT)
    .option("checkpointLocation", STREAM_CKPT)
    .trigger(processingTime=CFG["streaming"]["trigger_interval"])
    .start()
)

timeout = CFG["streaming"].get("timeout_seconds", 90)
print(f"Streaming query running for {timeout}s ...")
query.awaitTermination(timeout=timeout)

progress = query.lastProgress
if progress:
    with open(f"{proof}/query_progress.json", "w") as fh:
        json.dump(progress, fh, indent=2, default=str)
    print(json.dumps(progress, indent=2, default=str))
    log_metric("streaming", "windowed_agg", "num_input_rows",
               progress.get("numInputRows", 0), "lastProgress")
    log_metric("streaming", "windowed_agg", "processing_time_ms",
               progress.get("durationMs", {}).get("triggerExecution", None))

query.stop()
print(f"Streaming output: {parquet_size_mb(STREAM_OUT)} MB")

26/05/24 19:29:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming query running for 90s ...
{
  "id": "c8ccbbbc-5848-4c94-a2a3-af4934d23590",
  "runId": "e14f36a8-38e3-47eb-981d-070198ab334c",
  "name": null,
  "timestamp": "2026-05-24T17:30:30.000Z",
  "batchId": 3,
  "batchDuration": 5,
  "durationMs": {
    "triggerExecution": 5,
    "latestOffset": 4
  },
  "eventTime": {
    "watermark": "2023-12-31T17:45:42.000Z"
  },
  "stateOperators": [
    {
      "operatorName": "stateStoreSave",
      "numRowsTotal": 1,
      "numRowsUpdated": 0,
      "numRowsRemoved": 0,
      "allUpdatesTimeMs": 0,
      "allRemovalsTimeMs": 0,
      "commitTimeMs": 449,
      "memoryUsedBytes": 69704,
      "numRowsDroppedByWatermark": 0,
      "numShufflePartitions": 8,
      "numStateStoreInstances": 8,
      "customMetrics": {
        "stateOnCurrentVersionSizeBytes": 1608,
        "loadedMapCacheHitCount": 32,
        "loadedMapCacheMissCount": 0
      }
    }
  ],
  "sources": [
    {
      "description": "FileStreamSource[file:/home/maxence/Documents/D

26/05/24 19:30:54 WARN DAGScheduler: Failed to cancel job group e14f36a8-38e3-47eb-981d-070198ab334c. Cannot find active jobs for it.
26/05/24 19:30:54 WARN DAGScheduler: Failed to cancel job group e14f36a8-38e3-47eb-981d-070198ab334c. Cannot find active jobs for it.


---
## 5. Text Pipeline — Corpus → Inverted Index → Latency Benchmarks

Corpus: `callsign` + `dep_airport` + `arr_airport` concatenated as synthetic "flight text".  
Steps: tokenize → normalize → remove stop-words → inverted index.  
Comparison: Parquet vs CSV storage footprint + single-term query latency.

In [19]:
t0_text = time.time()

# ── Build text corpus from silver
df_s = spark.read.parquet(silver)

STOPWORDS = set(CFG["text"]["stopwords"])
stopwords_bc = spark.sparkContext.broadcast(STOPWORDS)

# Concatenate fields into a "document" per flight
corpus = (
    df_s
    .withColumn("text",
        F.concat_ws(" ",
            F.coalesce("callsign", F.lit("")),
            F.coalesce("dep_airport", F.lit("")),
            F.coalesce("arr_airport", F.lit("")),
            F.coalesce(F.col("flight_duration_sec").cast("string"), F.lit(""))
        )
    )
    .withColumn("doc_id", F.monotonically_increasing_id())
    .select("doc_id", "text")
)

n_docs = corpus.count()
print(f"Corpus documents: {n_docs:,}")

# ── Tokenize & normalize
tokens = (
    corpus
    .withColumn("text_clean", F.regexp_replace(F.lower(F.col("text")), r"[^a-z0-9\s]", ""))
    .withColumn("tokens", F.split("text_clean", r"\s+"))
    .select("doc_id", F.explode("tokens").alias("token"))
    .filter(F.length(F.col("token")) >= CFG["text"]["min_token_length"])
)

# ── Remove stop-words via UDF
not_stopword = F.udf(lambda t: t not in stopwords_bc.value, T.BooleanType())
tokens_clean = tokens.filter(not_stopword(F.col("token")))

# ── Build inverted index
inverted_index = (
    tokens_clean
    .groupBy("token")
    .agg(
        F.count("*").alias("term_freq"),
        F.countDistinct("doc_id").alias("doc_freq"),
        F.collect_list("doc_id").alias("doc_ids")
    )
    .orderBy(F.desc("term_freq"))
)

save_plan(inverted_index, f"{proof}/plan_index_build.txt")

TEXT_OUT = "outputs/project/text"
inverted_index.write.mode("overwrite").parquet(TEXT_OUT)

n_terms = inverted_index.count()
duration_index = round(time.time() - t0_text, 2)

print(f"Unique terms     : {n_terms:,}")
print(f"Index build time : {duration_index} s")
log_metric("text", "inverted_index", "unique_terms",     n_terms)
log_metric("text", "inverted_index", "build_duration_s", duration_index)

inverted_index.show(10, truncate=False)

Corpus documents: 1,000,000
  [PLAN] Saved → proof//plan_index_build.txt


Unique terms     : 190,718
Index build time : 18.31 s
  [METRIC] text/inverted_index: unique_terms = 190718  ()
  [METRIC] text/inverted_index: build_duration_s = 18.31  ()


IOPub data rate exceeded.                                                       
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [22]:
TEXT_CSV = "outputs/project/text_csv"

inverted_index \
    .withColumn("doc_ids", F.concat_ws(",", F.col("doc_ids"))) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(TEXT_CSV)

In [23]:
# ── Query latency benchmark (warm cache)
idx = spark.read.parquet(TEXT_OUT)
idx.cache()
idx.count()  # force cache materialisation

save_plan(idx.filter(F.col("token") == "lfpg"), f"{proof}/plan_query.txt")

print("\nQuery latency benchmarks:")
for term in CFG["text"]["query_terms"]:
    t_q = time.time()
    res = idx.filter(F.col("token") == term.lower()).collect()
    latency_ms = round((time.time() - t_q) * 1000, 1)
    n_docs_hit = len(res[0]["doc_ids"]) if res else 0
    slo_ok = "PASS" if latency_ms <= 2000 else "FAIL"
    print(f"  '{term}': {latency_ms} ms | docs={n_docs_hit} | {slo_ok}")
    log_metric("text", "query_latency", f"latency_ms_{term}", latency_ms,
               f"SLO ≤ 2000 ms → {slo_ok}")

  [PLAN] Saved → proof//plan_query.txt

Query latency benchmarks:
  'flight': 232.7 ms | docs=0 | PASS
  [METRIC] text/query_latency: latency_ms_flight = 232.7  (SLO ≤ 2000 ms → PASS)
  'airport': 180.2 ms | docs=0 | PASS
  [METRIC] text/query_latency: latency_ms_airport = 180.2  (SLO ≤ 2000 ms → PASS)
  'delay': 126.0 ms | docs=0 | PASS
  [METRIC] text/query_latency: latency_ms_delay = 126.0  (SLO ≤ 2000 ms → PASS)
  'arrival': 82.3 ms | docs=0 | PASS
  [METRIC] text/query_latency: latency_ms_arrival = 82.3  (SLO ≤ 2000 ms → PASS)
  'departure': 48.1 ms | docs=0 | PASS
  [METRIC] text/query_latency: latency_ms_departure = 48.1  (SLO ≤ 2000 ms → PASS)


---
## 6. Iterative Workload — KMeans Clustering Sweep

**Choice: Clustering** (stated explicitly per project rules).

**Features:** `flight_duration_sec`, `dep_horiz_m`, `dep_vert_m`, `arr_horiz_m`, `arr_vert_m`

**Sweep:** k ∈ {3, 5, 7, 10, 15} × seeds {42, 123, 456} → silhouette mean ± std.  
**Partitioning experiment:** compare shuffle cost before/after `repartition` optimization.

**Bonus:** Drift detection — compare cluster assignments across two time windows.

In [24]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

df_s = spark.read.parquet(silver)

FEATURE_COLS = [
    "flight_duration_sec", "dep_horiz_m", "dep_vert_m",
    "arr_horiz_m", "arr_vert_m", "dep_cand", "arr_cand"
]

# ── Drop nulls on features
df_feat = df_s.select(["icao24", "firstseen", "day"] + FEATURE_COLS).dropna()

# ── Assemble & scale
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="raw_features")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features",
                            withMean=True, withStd=True)

df_assembled = assembler.transform(df_feat)
scaler_model = scaler.fit(df_assembled)
df_scaled    = scaler_model.transform(df_assembled).select("icao24", "firstseen", "day", "features")

# ── Cache for sweep
df_scaled.cache()
n_feat = df_scaled.count()
print(f"Feature rows: {n_feat:,}")
log_metric("clustering", "features", "row_count", n_feat)

Feature rows: 1,000,000
  [METRIC] clustering/features: row_count = 1000000  ()


In [26]:
# ── KMeans sweep: k × seed
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette")

k_values = CFG["iterative"]["k_values"]
seeds    = CFG["iterative"]["seeds"]
results  = []   # (k, seed, silhouette, duration)

save_plan(df_scaled, f"{proof}/plan_iterative_before.txt")

for k in k_values:
    sils = []
    for seed in seeds:
        t_k = time.time()
        km  = KMeans(k=k, seed=seed, maxIter=CFG["iterative"]["max_iter"],
                     featuresCol="features", predictionCol="prediction")
        model    = km.fit(df_scaled)
        preds    = model.transform(df_scaled)
        sil      = evaluator.evaluate(preds)
        dur      = round(time.time() - t_k, 2)
        cost     = model.summary.trainingCost
        sils.append(sil)
        results.append({"k": k, "seed": seed, "silhouette": round(sil, 4),
                        "cost": round(cost, 2), "duration_s": dur})
        print(f"  k={k:2d} seed={seed}: silhouette={sil:.4f} cost={cost:.0f} dur={dur}s")
        log_metric("clustering", f"k{k}_seed{seed}", "silhouette", round(sil, 4))
        log_metric("clustering", f"k{k}_seed{seed}", "cost",       round(cost, 2))

    import statistics
    mean_sil = round(statistics.mean(sils), 4)
    std_sil  = round(statistics.stdev(sils) if len(sils) > 1 else 0, 4)
    print(f"  k={k}: mean_silhouette={mean_sil} ± {std_sil}")
    log_metric("clustering", f"k{k}", "mean_silhouette", mean_sil, f"std={std_sil}")

# ── Summary
results_df = pd.DataFrame(results)
best_row   = results_df.loc[results_df["silhouette"].idxmax()]
print(f"\nBest config: k={int(best_row.k)}, seed={int(best_row.seed)}, "
      f"silhouette={best_row.silhouette}")
slo_ok = "PASS" if best_row.silhouette >= CFG["slos"]["clustering_silhouette_min"] else "FAIL"
print(f"Silhouette SLO (≥ {CFG['slos']['clustering_silhouette_min']}): {slo_ok}")
log_metric("clustering", "best", "best_silhouette", best_row.silhouette,
           f"k={int(best_row.k)} {slo_ok}")
print(results_df.to_string(index=False))

  [PLAN] Saved → proof//plan_iterative_before.txt
  k= 3 seed=42: silhouette=0.1649 cost=5794392 dur=8.51s
  [METRIC] clustering/k3_seed42: silhouette = 0.1649  ()
  [METRIC] clustering/k3_seed42: cost = 5794392.13  ()
  k= 3 seed=123: silhouette=0.1658 cost=5795539 dur=7.76s
  [METRIC] clustering/k3_seed123: silhouette = 0.1658  ()
  [METRIC] clustering/k3_seed123: cost = 5795539.23  ()
  k= 3 seed=456: silhouette=0.1647 cost=5796480 dur=7.85s
  [METRIC] clustering/k3_seed456: silhouette = 0.1647  ()
  [METRIC] clustering/k3_seed456: cost = 5796480.27  ()
  k=3: mean_silhouette=0.1652 ± 0.0006
  [METRIC] clustering/k3: mean_silhouette = 0.1652  (std=0.0006)
  k= 5 seed=42: silhouette=0.1712 cost=5160325 dur=8.58s
  [METRIC] clustering/k5_seed42: silhouette = 0.1712  ()
  [METRIC] clustering/k5_seed42: cost = 5160325.33  ()
  k= 5 seed=123: silhouette=0.1707 cost=5163226 dur=9.5s
  [METRIC] clustering/k5_seed123: silhouette = 0.1707  ()
  [METRIC] clustering/k5_seed123: cost = 5163225.

  k= 7 seed=42: silhouette=0.1761 cost=4721068 dur=13.21s
  [METRIC] clustering/k7_seed42: silhouette = 0.1761  ()
  [METRIC] clustering/k7_seed42: cost = 4721068.0  ()


  k= 7 seed=123: silhouette=0.1748 cost=4723942 dur=11.07s
  [METRIC] clustering/k7_seed123: silhouette = 0.1748  ()
  [METRIC] clustering/k7_seed123: cost = 4723941.9  ()


  k= 7 seed=456: silhouette=0.1770 cost=4716073 dur=10.98s
  [METRIC] clustering/k7_seed456: silhouette = 0.177  ()
  [METRIC] clustering/k7_seed456: cost = 4716072.99  ()
  k=7: mean_silhouette=0.176 ± 0.0011
  [METRIC] clustering/k7: mean_silhouette = 0.176  (std=0.0011)


  k=10 seed=42: silhouette=0.1793 cost=4267815 dur=11.68s
  [METRIC] clustering/k10_seed42: silhouette = 0.1793  ()
  [METRIC] clustering/k10_seed42: cost = 4267814.95  ()


  k=10 seed=123: silhouette=0.1751 cost=4281883 dur=11.72s
  [METRIC] clustering/k10_seed123: silhouette = 0.1751  ()
  [METRIC] clustering/k10_seed123: cost = 4281883.26  ()


  k=10 seed=456: silhouette=0.1781 cost=4275130 dur=11.68s
  [METRIC] clustering/k10_seed456: silhouette = 0.1781  ()
  [METRIC] clustering/k10_seed456: cost = 4275129.9  ()
  k=10: mean_silhouette=0.1775 ± 0.0021
  [METRIC] clustering/k10: mean_silhouette = 0.1775  (std=0.0021)


  k=15 seed=42: silhouette=0.1941 cost=3712537 dur=13.24s
  [METRIC] clustering/k15_seed42: silhouette = 0.1941  ()
  [METRIC] clustering/k15_seed42: cost = 3712536.82  ()


  k=15 seed=123: silhouette=0.1937 cost=3719437 dur=13.46s
  [METRIC] clustering/k15_seed123: silhouette = 0.1937  ()
  [METRIC] clustering/k15_seed123: cost = 3719437.11  ()


  k=15 seed=456: silhouette=0.1970 cost=3699694 dur=13.49s
  [METRIC] clustering/k15_seed456: silhouette = 0.197  ()
  [METRIC] clustering/k15_seed456: cost = 3699694.32  ()
  k=15: mean_silhouette=0.195 ± 0.0018
  [METRIC] clustering/k15: mean_silhouette = 0.195  (std=0.0018)

Best config: k=15, seed=456, silhouette=0.197
Silhouette SLO (≥ 0.25): FAIL
  [METRIC] clustering/best: best_silhouette = 0.197  (k=15 FAIL)
 k  seed  silhouette       cost  duration_s
 3    42      0.1649 5794392.13        8.51
 3   123      0.1658 5795539.23        7.76
 3   456      0.1647 5796480.27        7.85
 5    42      0.1712 5160325.33        8.58
 5   123      0.1707 5163225.76        9.50
 5   456      0.1687 5157895.78       10.95
 7    42      0.1761 4721068.00       13.21
 7   123      0.1748 4723941.90       11.07
 7   456      0.1770 4716072.99       10.98
10    42      0.1793 4267814.95       11.68
10   123      0.1751 4281883.26       11.72
10   456      0.1781 4275129.90       11.68
15    42

In [27]:
# ── Partitioning experiment: before vs after repartition
best_k = int(best_row.k)

# BEFORE — default shuffle partitions
t_before = time.time()
km_before = KMeans(k=best_k, seed=42, maxIter=10, featuresCol="features")
km_before.fit(df_scaled)
dur_before = round(time.time() - t_before, 2)

# AFTER — explicit repartition to reduce shuffle overhead
df_repartitioned = df_scaled.repartition(4)
df_repartitioned.cache()
df_repartitioned.count()

save_plan(df_repartitioned, f"{proof}/plan_iterative_after.txt")

t_after = time.time()
km_after = KMeans(k=best_k, seed=42, maxIter=10, featuresCol="features")
km_after.fit(df_repartitioned)
dur_after = round(time.time() - t_after, 2)

print(f"Before repartition : {dur_before}s")
print(f"After repartition  : {dur_after}s")
print(f"Speedup            : {round(dur_before/dur_after, 2)}x")

log_metric("clustering", "partition_exp", "before_dur_s", dur_before)
log_metric("clustering", "partition_exp", "after_dur_s",  dur_after)
log_metric("clustering", "partition_exp", "speedup",      round(dur_before/dur_after, 2))

  [PLAN] Saved → proof//plan_iterative_after.txt
Before repartition : 5.27s
After repartition  : 5.84s
Speedup            : 0.9x
  [METRIC] clustering/partition_exp: before_dur_s = 5.27  ()
  [METRIC] clustering/partition_exp: after_dur_s = 5.84  ()
  [METRIC] clustering/partition_exp: speedup = 0.9  ()


---
## 7. LLM Data Readiness

Curated corpus for RAG / LLM fine-tuning on aviation domain.  
**Schema:** `doc_id`, `text`, `source`, `version`, `curated_at`, `content_hash`, `dep_airport`, `arr_airport`  
**Quality filters applied:**
1. NULL text removed
2. Text length ≥ 50 chars
3. Deduplication on `content_hash` (xxhash64)
4. Language signal: ASCII ratio ≥ 0.95 (aviation codes are ASCII)

**Data Card** documented below.

In [29]:
t0_llm = time.time()

llm_rules = CFG.get("llm", {})
MIN_LEN   = llm_rules.get("min_text_length", 50)

df_s = spark.read.parquet(silver)

# ── Build rich text representation
df_text = (
    df_s
    .withColumn("text",
        F.format_string(
            "Flight %s operated aircraft %s departed %s bound for %s. "
            "Departure distance from airport: %d m horizontal, %d m vertical. "
            "Arrival distance from airport: %d m horizontal, %d m vertical. "
            "Total flight duration: %d seconds.",
            F.col("callsign"), F.col("icao24"),
            F.col("dep_airport"), F.col("arr_airport"),
            F.col("dep_horiz_m"), F.col("dep_vert_m"),
            F.col("arr_horiz_m"), F.col("arr_vert_m"),
            F.col("flight_duration_sec")
        )
    )
    .withColumn("doc_id", F.concat_ws("_", F.col("icao24"), F.col("firstseen").cast("string")))
    .select("doc_id", "text", "dep_airport", "arr_airport", "event_ts")
)

n_before = df_text.count()

# ── Quality filter 1: not null
df_llm = df_text.filter(F.col("text").isNotNull())
# ── Quality filter 2: minimum length
df_llm = df_llm.filter(F.length(F.col("text")) >= MIN_LEN)
# ── Quality filter 3: ASCII ratio ≥ 0.95
ascii_ratio_udf = F.udf(
    lambda s: sum(ord(c) < 128 for c in s) / len(s) if s else 0.0,
    T.DoubleType()
)
df_llm = df_llm.withColumn("ascii_ratio", ascii_ratio_udf(F.col("text")))
df_llm = df_llm.filter(F.col("ascii_ratio") >= 0.95)
# ── Quality filter 4: deduplicate by content hash
df_llm = (
    df_llm
    .withColumn("content_hash", F.xxhash64(F.col("text")))
    .dropDuplicates(["content_hash"])
)

# ── Add metadata
df_llm = (
    df_llm
    .withColumn("source",     F.lit(CFG["llm"]["source"]))
    .withColumn("version",    F.lit(CFG["llm"]["schema_version"]))
    .withColumn("curated_at", F.current_timestamp())
    .select("doc_id", "text", "dep_airport", "arr_airport",
            "event_ts", "source", "version", "curated_at", "content_hash")
)

save_plan(df_llm, f"{proof}/plan_llm_curation.txt")

LLM_OUT = "outputs/project/llm_ready"
df_llm.write.mode("overwrite").parquet(LLM_OUT)

n_after    = df_llm.count()
pass_rate  = round(n_after / n_before, 4) if n_before > 0 else 0
dur_llm    = round(time.time() - t0_llm, 2)
size_llm   = parquet_size_mb(LLM_OUT)

slo_ok = "PASS" if pass_rate >= CFG["slos"]["llm_quality_pass_rate_min"] else "FAIL"
print(f"Before filters   : {n_before:,}")
print(f"After filters    : {n_after:,}")
print(f"Pass rate        : {pass_rate:.2%}  (SLO ≥ 80% → {slo_ok})")
print(f"LLM Parquet MB   : {size_llm}")
print(f"Duration         : {dur_llm} s")

log_metric("llm", "curation", "n_before",     n_before)
log_metric("llm", "curation", "n_after",      n_after)
log_metric("llm", "curation", "pass_rate",    pass_rate, f"SLO ≥ 0.80 → {slo_ok}")
log_metric("llm", "curation", "parquet_mb",   size_llm)
log_metric("llm", "curation", "duration_sec", dur_llm)

df_llm.printSchema()
df_llm.show(3, truncate=80)

  [PLAN] Saved → proof//plan_llm_curation.txt


Before filters   : 1,000,000
After filters    : 1,000,000
Pass rate        : 100.00%  (SLO ≥ 80% → PASS)
LLM Parquet MB   : 97.68
Duration         : 20.83 s
  [METRIC] llm/curation: n_before = 1000000  ()
  [METRIC] llm/curation: n_after = 1000000  ()
  [METRIC] llm/curation: pass_rate = 1.0  (SLO ≥ 0.80 → PASS)
  [METRIC] llm/curation: parquet_mb = 97.68  ()
  [METRIC] llm/curation: duration_sec = 20.83  ()
root
 |-- doc_id: string (nullable = false)
 |-- text: string (nullable = false)
 |-- dep_airport: string (nullable = true)
 |-- arr_airport: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- source: string (nullable = false)
 |-- version: string (nullable = false)
 |-- curated_at: timestamp (nullable = false)
 |-- content_hash: long (nullable = false)



+-----------------+--------------------------------------------------------------------------------+-----------+-----------+-------------------+-----------------------------------+-------+--------------------------+--------------------+
|           doc_id|                                                                            text|dep_airport|arr_airport|           event_ts|                             source|version|                curated_at|        content_hash|
+-----------------+--------------------------------------------------------------------------------+-----------+-----------+-------------------+-----------------------------------+-------+--------------------------+--------------------+
|918ee7_1691575673|Flight ACA6728 operated aircraft 918ee7 departed LIRF bound for EKCH. Departu...|       LIRF|       EKCH|2023-08-09 12:07:53|OpenSky Network - ADS-B flight data|   v1.0|2026-05-24 19:42:05.944518|-9223277305577859286|
|343a22_1676342415|Flight RYR3052 operated aircraft 

In [30]:
# ── Data Card
DATA_CARD = f"""
# Data Card — Aviation LLM-Ready Dataset

| Field           | Value |
|-----------------|-------|
| Source          | {CFG['llm']['source']} |
| Track           | D — Aviation / OpenSky |
| Raw rows        | {n_before:,} |
| Curated rows    | {n_after:,} |
| Pass rate       | {pass_rate:.2%} |
| Schema version  | {CFG['llm']['schema_version']} |
| Intended use    | {CFG['llm']['intended_use']} |
| Filters applied | NULL removal, min_length={MIN_LEN}, ascii_ratio≥0.95, xxhash64 dedup |
| Output format   | Parquet (snappy) |
| Output size     | {size_llm} MB |
| Curated at      | {datetime.datetime.now().isoformat()} |

## Schema
- `doc_id`       STRING  — icao24_firstseen composite key
- `text`         STRING  — natural language flight description
- `dep_airport`  STRING  — ICAO departure airport code
- `arr_airport`  STRING  — ICAO arrival airport code
- `event_ts`     TIMESTAMP — flight start timestamp
- `source`       STRING  — dataset source identifier
- `version`      STRING  — schema version tag
- `curated_at`   TIMESTAMP — curation run timestamp
- `content_hash` LONG    — xxhash64 for deduplication
"""

with open(f"{LLM_OUT}/data_card.md", "w") as fh:
    fh.write(DATA_CARD)
print(DATA_CARD)


# Data Card — Aviation LLM-Ready Dataset

| Field           | Value |
|-----------------|-------|
| Source          | OpenSky Network - ADS-B flight data |
| Track           | D — Aviation / OpenSky |
| Raw rows        | 1,000,000 |
| Curated rows    | 1,000,000 |
| Pass rate       | 100.00% |
| Schema version  | v1.0 |
| Intended use    | RAG / LLM fine-tuning for aviation domain |
| Filters applied | NULL removal, min_length=50, ascii_ratio≥0.95, xxhash64 dedup |
| Output format   | Parquet (snappy) |
| Output size     | 97.68 MB |
| Curated at      | 2026-05-24T19:42:26.757689 |

## Schema
- `doc_id`       STRING  — icao24_firstseen composite key
- `text`         STRING  — natural language flight description
- `dep_airport`  STRING  — ICAO departure airport code
- `arr_airport`  STRING  — ICAO arrival airport code
- `event_ts`     TIMESTAMP — flight start timestamp
- `source`       STRING  — dataset source identifier
- `version`      STRING  — schema version tag
- `curated_at`   TI

---
## 8. Evidence — Plans, Metrics, Spark UI

Save all query plans. Finalize `project_metrics_log.csv`. Print Spark UI instructions.

In [31]:
import os

pathlib.Path(proof).mkdir(parents=True, exist_ok=True)

# ── List all proof files
proof_files = sorted(pathlib.Path(proof).glob("*"))
print(f"Proof artifacts ({len(proof_files)}):")
for pf in proof_files:
    print(f"  {pf.name}  ({pf.stat().st_size:,} bytes)")

# ── Print metrics summary
print(f"\nMetrics log: {METRICS_FILE}")
if pathlib.Path(METRICS_FILE).exists():
    df_metrics = pd.read_csv(METRICS_FILE)
    print(df_metrics.to_string(index=False))

# ── Spark UI screenshot reminder
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ACTION REQUIRED: Capture Spark UI screenshots
  URL: {spark.sparkContext.uiWebUrl}

  Pages to screenshot:
  • Jobs → save as proof/spark_ui_jobs.png
  • Stages → save as proof/spark_ui_stages.png
  • SQL/DataFrame tab → save as proof/spark_ui_sql.png
  • Streaming tab → save as proof/spark_ui_streaming.png
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

Proof artifacts (10):
  plan_gold_aircraft.txt  (4,880 bytes)
  plan_gold_daily.txt  (5,742 bytes)
  plan_gold_routes.txt  (6,132 bytes)
  plan_index_build.txt  (7,914 bytes)
  plan_iterative_after.txt  (13,376 bytes)
  plan_iterative_before.txt  (7,549 bytes)
  plan_llm_curation.txt  (12,412 bytes)
  plan_query.txt  (1,756 bytes)
  plan_silver.txt  (24,922 bytes)
  query_progress.json  (1,522 bytes)

Metrics log: project_metrics_log.csv
         run_id      stage           task          metric_name  metric_value                       notes                  timestamp
20260524_191734        etl         bronze            row_count  1000000.0000               raw ingestion 2026-05-24T19:18:08.752976
20260524_191734        etl         bronze           parquet_mb       44.7200    bronze Parquet footprint 2026-05-24T19:18:08.760526
20260524_191734        etl         bronze               csv_mb       79.3900            raw CSV baseline 2026-05-24T19:18:08.766045
20260524_191734        etl    

In [32]:
# ── Final SLO summary
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("FINAL SLO SUMMARY")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

if pathlib.Path(METRICS_FILE).exists():
    m = pd.read_csv(METRICS_FILE)

    def get_val(stage, metric):
        row = m[(m.stage == stage) & (m.metric_name == metric)]
        return float(row.metric_value.iloc[0]) if len(row) > 0 else None

    checks = [
        ("Storage ratio vs raw CSV ≤ 0.60",
         get_val("etl", "storage_ratio"), 0.60, "<="),
        ("LLM pass rate ≥ 0.80",
         get_val("llm", "pass_rate"), 0.80, ">="),
    ]
    # clustering best silhouette
    sil_row = m[m.metric_name == "best_silhouette"]
    if len(sil_row) > 0:
        checks.append(("Best silhouette ≥ 0.25",
                        float(sil_row.metric_value.iloc[0]), 0.25, ">="))

    for label, val, target, op in checks:
        if val is None:
            status = "N/A"
        elif op == "<=":
            status = "✓ PASS" if val <= target else "✗ FAIL"
        else:
            status = "✓ PASS" if val >= target else "✗ FAIL"
        print(f"  {status}  {label}  (got {val})")

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL SLO SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓ PASS  Storage ratio vs raw CSV ≤ 0.60  (got 0.449)
  ✓ PASS  LLM pass rate ≥ 0.80  (got 1.0)
  ✗ FAIL  Best silhouette ≥ 0.25  (got 0.197)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [33]:
spark.stop()
print("Spark session stopped. Pipeline complete.")

Spark session stopped. Pipeline complete.
